# Week 7: QLoRA Fine-tuning Mistral-7B

**Runtime → Change runtime type → T4 GPU**

Upload train.json and validation.json via the Files panel (left sidebar folder icon).

In [ ]:
# STEP 1: Install dependencies (~3 minutes)
!pip install -q transformers==4.44.0 peft==0.12.0 trl==0.10.1 bitsandbytes==0.43.3 accelerate==0.34.2 datasets==3.0.0 huggingface_hub sentencepiece
print("Done!")

In [ ]:
# STEP 2: Login to HuggingFace
from huggingface_hub import login

HF_TOKEN = "hf_YOUR_TOKEN_HERE"   # <- paste your HuggingFace WRITE token
HF_USERNAME = "your-username"      # <- your HuggingFace username
MODEL_OUTPUT_NAME = "legal-mistral-7b"

login(token=HF_TOKEN)
print(f"Logged in! Model will be: {HF_USERNAME}/{MODEL_OUTPUT_NAME}")

In [ ]:
# STEP 3: Upload training data (a file picker will appear below)
from google.colab import files
import json

print("Please upload train.json and validation.json when prompted...")
uploaded = files.upload()  # opens file picker — select both files

train_data = json.loads(uploaded['train.json'].decode('utf-8'))
val_data   = json.loads(uploaded['validation.json'].decode('utf-8'))

print(f"Train: {len(train_data)} examples")
print(f"Val  : {len(val_data)} examples")
print(f"Sample: {train_data[0]['instruction'][:60]}")

In [ ]:
# STEP 4: Convert to HuggingFace Dataset (Alpaca format)
from datasets import Dataset

def format_alpaca(example):
    return {
        "text": (
            f"### Instruction:
{example['instruction']}

"
            f"### Input:
{example['input']}

"
            f"### Response:
{example['output']}"
        )
    }

train_dataset = Dataset.from_list([format_alpaca(ex) for ex in train_data])
val_dataset   = Dataset.from_list([format_alpaca(ex) for ex in val_data])

print(f"Train: {len(train_dataset)} rows")
print(f"Val  : {len(val_dataset)} rows")
print("
Sample:")
print(train_dataset[0]["text"][:400])

In [ ]:
# STEP 5: Load Mistral-7B in 4-bit (QLoRA)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model in 4-bit (~5 minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False
model.config.pretraining_tp = 1
print("Model loaded!")

In [ ]:
# STEP 6: Configure LoRA adapters
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("LoRA configured!")

In [ ]:
# STEP 7: Fine-tune with SFTTrainer (~2 hours on T4)
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=200,
    logging_steps=50,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="none",
    evaluation_strategy="steps",
    eval_steps=200,
    load_best_model_at_end=True
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
    packing=False
)

print("Starting fine-tuning... (~2 hours on T4)")
trainer.train()
print("Fine-tuning complete!")

In [ ]:
# STEP 8: Push fine-tuned model to HuggingFace Hub
repo_id = f"{HF_USERNAME}/{MODEL_OUTPUT_NAME}"

print(f"Pushing to {repo_id}...")
trainer.model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

print(f"
Done! Copy this for your local notebook:")
print(f"  MODEL_ID = "{repo_id}"")